In [ ]:
#!/usr/bin/env python3
"""Loads the ESMC embeddings of step 03: one protein x channel matrix per label for the chosen MODEL.

Paths come from the extract_reps_biohub section of configs/param_configs.yaml; labels are named and ordered as in the query table (exact_species). The 6B matrices are large (about 10 KB per protein), so only one model is loaded.
"""
###################################### DEPENDENCIES_AND_CONSTANTS
from IPython.display import display
import numpy as np
import pandas as pd
from pathlib import Path
from pyprojroot import find_root, has_file
import yaml
DEFAULT_QUERY_TSV = "configs/query_configs.tsv"
YAML_PATH = "configs/param_configs.yaml"
MODEL = "esmc_6b"
###################################### CONFIG
PROJECT_ROOT = find_root(has_file(".here"), start=Path.cwd()).resolve()
config = yaml.safe_load((PROJECT_ROOT / YAML_PATH).read_text()) or {}
query_table = pd.read_csv(PROJECT_ROOT / config.get("dw_uniprot_seqs", {}).get("query_table", DEFAULT_QUERY_TSV), sep="\t", dtype=str)
species_names = dict(zip(query_table["label"], query_table["exact_species"].fillna(query_table["label"])))
embeddings_dir = PROJECT_ROOT / config["extract_reps_biohub"]["embeddings_dir"] / MODEL
print(f"[PYTHON-INFO] {MODEL}: {sorted(p.name for p in embeddings_dir.glob('*_reps.npz'))}")

In [ ]:
###################################### LOAD
embeddings, protein_ids = {}, {}
for lbl in species_names:
    path = embeddings_dir / f"{lbl}_reps.npz"
    if not path.exists(): continue
    with np.load(path) as npz: embeddings[lbl], protein_ids[lbl] = npz["embeddings"], npz["ids"].astype(str)
display(pd.DataFrame({"species": species_names, "proteins": {l: e.shape[0] for l, e in embeddings.items()}, "channels": {l: e.shape[1] for l, e in embeddings.items()}, "MB": {l: round(e.nbytes / 1e6, 1) for l, e in embeddings.items()}}))